In [1]:
import polars as pl
import os

def generate_test_csv(input_path='./kaggle/train.csv', output_path='./kaggle/test.csv', days_to_keep=180):
    # 1. Read the training data
    print(f"Reading {input_path}...")
    df = pl.read_csv(input_path)
    
    # Ensure sorted by date_id to strictly respect time ordering for the shift
    if "date_id" in df.columns:
        df = df.sort("date_id")
    
    # 2. Create the Lagged Columns (Shifted by 1)
    # The prompt asks to "rename" and "shift". In practice for test sets, 
    # this means the value available at time T is the return from time T-1.
    df = df.with_columns([
        pl.col("forward_returns").shift(1).alias("lagged_forward_returns"),
        pl.col("risk_free_rate").shift(1).alias("lagged_risk_free_rate"),
        pl.col("market_forward_excess_returns").shift(1).alias("lagged_market_forward_excess_returns"),
        pl.lit(1).alias("is_scored") # Set is_scored to 1 as requested
    ])
    
    # 3. Filter for the last 180 days
    # We do this AFTER shifting to ensure the first row of the test set 
    # correctly gets the lagged value from the day immediately preceding it.
    unique_dates = df["date_id"].unique().sort()
    cutoff_date = unique_dates.tail(days_to_keep).head(1).item()
    
    test_df = df.filter(pl.col("date_id") >= cutoff_date)
    
    # 4. Clean up columns
    # We remove the original forward-looking targets because a real test.csv 
    # would not contain the answers for the current day.
    targets_to_remove = [
        "forward_returns", 
        "risk_free_rate", 
        "market_forward_excess_returns"
    ]
    
    # Select only columns that are NOT the original targets
    # This keeps all features (M1, P1, etc), date_id, and the new lagged cols
    final_cols = [c for c in test_df.columns if c not in targets_to_remove]
    test_df = test_df.select(final_cols)
    
    # 5. Write to CSV
    print(f"Writing {test_df.height} rows to {output_path}...")
    test_df.write_csv(output_path)
    print("Done.")
    
    # Verification print
    print("\nColumns in test.csv:")
    print(test_df.columns)
    print("\nSample Data:")
    print(test_df.head(5))


# Adjust path if your train.csv is in a different location
train_path = './kaggle/train.csv' 

#generate_test_csv(train_path)


In [2]:
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

import polars as pl
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Input
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# --- Configuration ---
MAX_HISTORY_LEN = 400
RNN_LOOKBACK = 60
TRADING_DAYS_PER_YR = 252



2025-12-15 17:42:52.195137: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-15 17:42:52.226032: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-15 17:42:53.703067: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

import polars as pl
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Input
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# --- Configuration ---
MAX_HISTORY_LEN = 400
RNN_LOOKBACK = 60
TRADING_DAYS_PER_YR = 252



def generate_features(df: pl.DataFrame) -> pl.DataFrame:
  """Generates new features from the base polars dataframe.
    
      Available Feature Categories:
      - D* (Dummy/Binary features): 9 columns (D1-D9)
      - E* (Macro Economic features): 20 columns (E1-E20)
      - I* (Interest Rate features): 9 columns (I1-I9)
      - M* (Market Dynamics/Technical features): 18 columns (M1-M18)
      - P* (Price/Valuation features): 13 columns (P1-P13)
      - S* (Sentiment features): 12 columns (S1-S12)
      - V* (Volatility features): 13 columns (V1-V13)
    
        Top 20 Most Positively Correlated Features with Target:
        ['V13', 'M1', 'S5', 'D1', 'D2', 'M2', 'V10', 'V7', 'S12', 'S6', 'M17', 'D8', 'E19', 'D4', 'D6', 'V9', 'M3', 'D7', 'E9', 'V6']
    
        Top 19 Most Negatively Correlated Features with Target:
        ['M4', 'S2', 'P8', 'E7', 'E11', 'E12', 'M12', 'I2', 'P7', 'P5', 'P10', 'P12', 'M8', 'S3', 'S7', 'P11', 'P3', 'E13', 'I1']
    
        Do not look into future, e.g. do not use negative lags or backward fill.
  """
  new_features = pl.DataFrame({
    # --- Retained Features from v1/v2 ---
    'feat_M1_roll_median_10': df['M1'].rolling_median(window_size=10),
    'feat_V13_roll_median_10': df['V13'].rolling_median(window_size=10),
    'feat_S5_roll_median_5': df['S5'].rolling_median(window_size=5),
    'feat_D1_roll_mean_10': df['D1'].rolling_mean(window_size=10),
    'feat_D2_roll_std_5': df['D2'].rolling_std(window_size=5),
    'feat_D4_roll_mean_20': df['D4'].rolling_mean(window_size=20),
    'feat_D6_roll_mean_10': df['D6'].rolling_mean(window_size=10),
    'feat_D7_roll_std_5': df['D7'].rolling_std(window_size=5),
    'feat_M1_roll_10_vs_P1_roll_10_diff': df['M1'].rolling_mean(window_size=10) - df['P1'].rolling_mean(window_size=10),
    'feat_V13_roll_10_vs_M1_roll_10_diff': df['V13'].rolling_mean(window_size=10) - df['M1'].rolling_mean(window_size=10),
    'feat_S5_roll_5_vs_D1_roll_5_diff': df['S5'].rolling_mean(window_size=5) - df['D1'].rolling_mean(window_size=5),
    'feat_S12_lag_3': df['S12'].shift(3),
    'feat_V10_lag_2': df['V10'].shift(2),
    'feat_E19_lag_1': df['E19'].shift(1),
    'feat_M1_roll_max_5': df['M1'].rolling_max(window_size=5),
    'feat_M1_roll_min_5': df['M1'].rolling_min(window_size=5),
    'feat_V13_roll_max_10': df['V13'].rolling_max(window_size=10),
    'feat_V13_roll_min_10': df['V13'].rolling_min(window_size=10),
    'feat_M1_diff_1_1': df['M1'] - df['M1'].shift(1),
    'feat_V13_diff_1_1': df['V13'] - df['V13'].shift(1),
    'feat_S5_diff_1_1': df['S5'] - df['S5'].shift(1),
    'feat_M1_roll_std_10': df['M1'].rolling_std(window_size=10),
    'feat_V13_roll_std_10': df['V13'].rolling_std(window_size=10),
    'feat_S5_roll_std_5': df['S5'].rolling_std(window_size=5),
    'feat_M4_lag_1': df['M4'].shift(1),
    'feat_P8_roll_mean_5': df['P8'].rolling_mean(window_size=5),
    'feat_E7_roll_std_10': df['E7'].rolling_std(window_size=10),
    'feat_M1_mul_V13_ratio_10': df['M1'].rolling_mean(10) / df['V13'].rolling_mean(10),
    'feat_D8_roll_mean_10': df['D8'].rolling_mean(window_size=10),
    'feat_E9_roll_std_10': df['E9'].rolling_std(window_size=10),
    'feat_V6_roll_diff_5': df['V6'].rolling_mean(window_size=5) - df['V6'].shift(5),
    'feat_V7_roll_mean_5': df['V7'].rolling_mean(window_size=5),
    'feat_V9_roll_mean_5': df['V9'].rolling_mean(window_size=5),
    'feat_M3_roll_std_10': df['M3'].rolling_std(window_size=10),
    'feat_M1_div_V13_10': df['M1'].rolling_mean(10) / df['V13'].rolling_mean(10),
    'feat_S5_div_D1_5': df['S5'].rolling_mean(5) / df['D1'].rolling_mean(5),
    'feat_P7_roll_std_5': df['P7'].rolling_std(window_size=5),
    'feat_I2_lag_2': df['I2'].shift(2),
    'feat_E11_roll_mean_10': df['E11'].rolling_mean(window_size=10),
    'feat_D8_diff_1': df['D8'] - df['D8'].shift(1),
    'feat_V7_diff_1': df['V7'] - df['V7'].shift(1),
    'feat_S12_roll_mean_5': df['S12'].rolling_mean(window_size=5),
    'feat_V10_roll_std_5': df['V10'].rolling_std(window_size=5),
    'feat_V10_roll_min_5': df['V10'].rolling_min(window_size=5),
    'feat_S12_roll_std_5': df['S12'].rolling_std(window_size=5),
    # Features added in v3 (V7, M3, E19, D8 momentum/stats, M1/S5 ratio)
    'feat_V7_roll_std_5': df['V7'].rolling_std(window_size=5),
    'feat_V7_roll_max_5': df['V7'].rolling_max(window_size=5),
    'feat_M3_roll_mean_5': df['M3'].rolling_mean(window_size=5),
    'feat_M3_roll_std_5': df['M3'].rolling_std(window_size=5),
    'feat_E19_roll_mean_5': df['E19'].rolling_mean(window_size=5),
    'feat_E19_diff_1': df['E19'] - df['E19'].shift(1),
    'feat_D8_roll_std_10': df['D8'].rolling_std(window_size=10),
    'feat_M1_div_S5_10': df['M1'].rolling_mean(10) / df['S5'].rolling_mean(10),
  })
  # --- New additions for v3 (Focus on negatively correlated features P8, P7, I2, E11, P5, P10, P12, P3) ---
  # P8 (Neg Corr)
  new_features = new_features.with_columns(
    (df['P8'].rolling_std(window_size=10).alias('feat_P8_roll_std_10')),
    (df['P8'].shift(5).alias('feat_P8_lag_5')),
    (df['P8'].rolling_sum(window_size=5).alias('feat_P8_roll_sum_5'))
  )
  # P7 (Neg Corr)
  new_features = new_features.with_columns(
    (df['P7'].rolling_mean(window_size=10).alias('feat_P7_roll_mean_10')),
    (df['P7'].shift(1).alias('feat_P7_lag_1')),
    (df['P7'].rolling_max(window_size=5).alias('feat_P7_roll_max_5'))
  )
  # I2 (Neg Corr)
  new_features = new_features.with_columns(
    (df['I2'].rolling_mean(window_size=5).alias('feat_I2_roll_mean_5')),
    (df['I2'].diff().alias('feat_I2_diff_1')),
    (df['I2'].rolling_min(window_size=5).alias('feat_I2_roll_min_5'))
  )
  # E11 (Neg Corr)
  new_features = new_features.with_columns(
    (df['E11'].rolling_std(window_size=5).alias('feat_E11_roll_std_5')),
    (df['E11'].shift(5).alias('feat_E11_lag_5')),
    (df['E11'].rolling_mean(window_size=5).alias('feat_E11_roll_mean_5'))
  )
  # Adding some cross-feature momentum (M1/V13 momentum vs M1/V13 current state)
  new_features = new_features.with_columns(
    ((df['M1'].rolling_mean(10) - df['M1'].shift(5)).alias('feat_M1_roll_10_momentum_5')),
    ((df['V13'].rolling_mean(10) - df['V13'].shift(5)).alias('feat_V13_roll_10_momentum_5')),
  )
  # --- New additions for v4 (Focus on Exponential Moving Averages) ---
  new_features = new_features.with_columns(
    (df['M1'].ewm_mean(alpha=0.3).alias('feat_M1_ewm_0.3')),
    (df['V13'].ewm_mean(alpha=0.3).alias('feat_V13_ewm_0.3')),
    (df['M1'].ewm_mean(alpha=0.4).alias('feat_M1_ewm_0.4')),
    (df['V13'].ewm_mean(alpha=0.4).alias('feat_V13_ewm_0.4')),
    (df['M1'].ewm_mean(alpha=0.5).alias('feat_M1_ewm_0.5')),
    (df['V13'].ewm_mean(alpha=0.5).alias('feat_V13_ewm_0.5')),
    (df['M1'].ewm_mean(alpha=0.6).alias('feat_M1_ewm_0.6')),
    (df['V13'].ewm_mean(alpha=0.6).alias('feat_V13_ewm_0.6'))
  )
  return new_features.with_columns(pl.all().forward_fill())




class OnlineStrategy:

    def __init__(self):
        self.history_df = pl.DataFrame()
        self.fitted = False
        self.feat_cols = [] 
        self.raw_cols = [] 
        
        # --- State for Continuity ---
        self.last_date_id = 0
        self.last_sp_index = 1.0
        self.raw_scaler = StandardScaler()
        self.scale_cols = [] # Columns that need pre-normalization
        
        # --- Storage for Retraining ---
        self.full_train_df = None
        self.last_train_date = -1

        # Volatility Control State
        self.pred_history = {}  # Format: {'Model_Name': [val1, val2, ...]}
        self.vol_target = 20
        self.vol_window = 20
        self.z_window = 60
        self.leverage_cap = 2.0
        # -------------------------

        # --- Models ---\n
        self.models = {
            'XGB_Std': xgb.XGBRegressor(n_estimators=2, max_depth=5, n_jobs=-1, random_state=42),
            'XGB_Rev': xgb.XGBRegressor(n_estimators=2, max_depth=5, n_jobs=-1, random_state=42),
            'LGB_Std': lgb.LGBMRegressor(n_estimators=2, max_depth=4, n_jobs=-1, random_state=42, verbose=-1),
            'LGB_Rev': lgb.LGBMRegressor(n_estimators=2, max_depth=4, n_jobs=-1, random_state=42, verbose=-1),
            'KNN_Std': KNeighborsRegressor(n_neighbors=40, n_jobs=-1),
            'KNN_Rev': KNeighborsRegressor(n_neighbors=40, n_jobs=-1),
            'Bagging_Std': BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=5), n_estimators=10, random_state=42, n_jobs=-1),
            'Bagging_Rev': BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=5), n_estimators=10, random_state=42, n_jobs=-1),
        }
        self.rnn_models = {}
        # self.rnn_names = ['LSTM_Std', 'LSTM_Rev', 'GRU_Std', 'GRU_Rev']
        self.rnn_names = [] # Disabled for speed in this snippet, re-enable if needed
        
        # Preprocessing for Final X (post-feature-gen)
        self.imputer = SimpleImputer(strategy='constant', fill_value=0)
        self.final_scaler = StandardScaler()


    # --- INSERT THESE NEW METHODS ---
    def _get_vol_scalar(self):
        """Calculates (Target_Vol / Current_Vol) based on history."""
        # Check for 'lagged_forward_returns' or 'lagged_market_forward_return'
        col_name = None
        if 'lagged_forward_returns' in self.history_df.columns:
            col_name = 'lagged_forward_returns'
        elif 'lagged_market_forward_return' in self.history_df.columns:
            col_name = 'lagged_market_forward_return'
            
        if col_name and self.history_df.height >= 5:
            # Get last N days
            returns = self.history_df[col_name].tail(self.vol_window).to_numpy()
            
            # FIX: Use keyword argument for 'nan' to avoid passing 0.0 as 'copy'
            returns = np.nan_to_num(returns, nan=0.0) 
            
            # Calculate Daily Vol
            daily_vol = np.std(returns)
            if daily_vol < 1e-6: daily_vol = 0.005 # Avoid div by zero
            
            # Annualize target to daily
            daily_target = self.vol_target / np.sqrt(TRADING_DAYS_PER_YR)
            
            scalar = daily_target / daily_vol
            return np.clip(scalar, 0, self.leverage_cap)
        return 1.0

    def _apply_vol_control(self, name, raw_pred, vol_scalar):
        """Online implementation of: Position = Vol_Scalar * Tanh(Z_Score(Pred))"""
        # 1. Update History
        if name not in self.pred_history:
            self.pred_history[name] = []
        self.pred_history[name].append(raw_pred)
        
        if len(self.pred_history[name]) > self.z_window:
            self.pred_history[name].pop(0)
            
        # 2. Calculate Stats (Z-Score)
        history_arr = np.array(self.pred_history[name])
        if len(history_arr) < 5:
            mu = np.mean(history_arr)
            sigma = np.std(history_arr) + 1e-6
        else:
            mu = np.mean(history_arr)
            sigma = np.std(history_arr) + 1e-6
            
        z_score = (raw_pred - mu) / sigma
        
        # 3. Tanh Activation & Scalar
        return np.clip(vol_scalar * np.tanh(z_score), 0.0, 2.0)
    # --------------------------------

    def _generate_features(self, df: pl.DataFrame) -> pl.DataFrame:
        """
        Generates complex features. 
        CRITICAL: Expects 'sp_index' to ALREADY exist and be continuous.
        CRITICAL: Expects raw feature columns (M1, P1, etc.) to ALREADY be normalized.
        """
        if 'date_id' in df.columns:
            df = df.sort('date_id')

        # 1. Technical Indicators (rely on sp_index)
        def calculate_rsi_expr(price_col, period=14):
            delta = price_col.diff()
            up = delta.clip(lower_bound=0)
            down = delta.clip(upper_bound=0).abs()
            roll_up = up.rolling_mean(period)
            roll_down = down.rolling_mean(period)
            rs = roll_up / (roll_down + 1e-9)
            return 100.0 - (100.0 / (1.0 + rs))

        df = df.with_columns([
            calculate_rsi_expr(pl.col('sp_index'), 14).alias('ind_rsi_14'),
            calculate_rsi_expr(pl.col('sp_index'), 42).alias('ind_rsi_42'),
            (pl.col('sp_index').rolling_mean(5) / (pl.col('sp_index') + 1e-9)).alias('ind_sma_5'),
            (pl.col('sp_index').rolling_mean(30) / (pl.col('sp_index') + 1e-9)).alias('ind_sma_30'),
            (pl.col('sp_index').rolling_mean(200) / (pl.col('sp_index') + 1e-9)).alias('ind_sma_200'),
            (pl.col('sp_index').rolling_std(50) / (pl.col('sp_index') + 1e-9)).alias('ind_std_10'),
            (pl.col('sp_index').rolling_std(50) / (pl.col('sp_index') + 1e-9)).alias('ind_std_50'),
            (pl.col('sp_index').shift(TRADING_DAYS_PER_YR) / (pl.col('sp_index') + 1e-9)).alias('ind_price_1y_ago'),
            (pl.col('sp_index').rolling_mean(20) / (pl.col('sp_index') + 1e-9)).alias('ind_bb_mid'),
            (pl.col('sp_index').rolling_std(20) / (pl.col('sp_index') + 1e-9)).alias('ind_bb_std'),
        ])
        
        df = df.with_columns([
            (pl.col('ind_bb_mid') + 2 * pl.col('ind_bb_std')).alias('ind_bb_upper'),
            (pl.col('ind_bb_mid') - 2 * pl.col('ind_bb_std')).alias('ind_bb_lower')
        ])

        # 2. Strategies
        sig_rsi_mr = pl.when(pl.col('ind_rsi_14') < 30).then(1.5).when(pl.col('ind_rsi_14') > 70).then(0.5).otherwise(0.8)
        sig_rsi_mom = pl.when(pl.col('ind_rsi_14') > 50).then(1.5).otherwise(0.6)
        sig_ma_cross = pl.when(pl.col('ind_sma_30') > pl.col('ind_sma_200')).then(1.5).otherwise(0.5)
        
        z_score = (1.0 - pl.col('ind_sma_30')) / (pl.col('ind_std_50') + 1e-9)
        sig_ma_dist = pl.when(z_score < -2.0).then(1.5).when(z_score > 2.0).then(0.5).otherwise(1.0)
        
        sig_bb_break = pl.when(1.0 > pl.col('ind_bb_upper')).then(1.5).when(1.0 < pl.col('ind_bb_lower')).then(0.5).otherwise(0.8)
        sig_tsmom = pl.when(1.0 > pl.col('ind_price_1y_ago')).then(1.5).otherwise(0.6)

        df = df.with_columns([
            sig_rsi_mr.fill_null(1.0).alias('strat_rsi_mr'),
            sig_rsi_mom.fill_null(1.0).alias('strat_rsi_mom'),
            sig_ma_cross.fill_null(1.0).alias('strat_ma_cross'),
            sig_ma_dist.fill_null(1.0).alias('strat_ma_dist'),
            sig_bb_break.fill_null(1.0).alias('strat_bb_break'),
            sig_tsmom.fill_null(1.0).alias('strat_tsmom'),
        ])

        # 3. Complex Features (Uses Pre-Normalized Data)
        # def safe_col(name): return pl.col(name) if name in df.columns else pl.lit(0.0)

        # new_features = df.with_columns([
        #     (safe_col('M1') * safe_col('V1')).alias('feat_M1_x_V1'),
        #     (safe_col('P1') + safe_col('E1')).alias('feat_P1_add_E1'),
        #     (safe_col('S1') - safe_col('I1')).alias('feat_S1_sub_I1'),
        #     safe_col('V2').rolling_mean(5).alias('feat_V2_roll_mean_5'),
        #     safe_col('V1').rolling_std(5).alias('feat_V1_roll_std_5'),
        #     safe_col('M1').rolling_mean(20).alias('feat_M1_roll_mean_20'),
        #     safe_col('M3').rolling_std(20).alias('feat_M3_roll_std_20'),
        #     safe_col('P1').rolling_max(10).alias('feat_P1_roll_max_10'),
        #     safe_col('P1').rolling_min(10).alias('feat_P1_roll_min_10'),
        #     (safe_col('M1').rolling_mean(5) - safe_col('M1').rolling_mean(20)).alias('feat_roll_diff_M1_5_20'),
        # ])
        
        # Cleanup
        add_new_features = generate_features(df)
        new_features = pl.concat([df, add_new_features], how='horizontal')
        new_features = new_features.fill_null(0).fill_nan(0)
        return new_features

    def _sanitize_numpy(self, X):
        return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    def _build_rnn(self, input_shape):

        for name in self.rnn_names:
            model = Sequential()
            model.add(Input(shape=input_shape))
            if 'LSTM' in name:
                model.add(LSTM(32, activation='tanh', return_sequences=False))
            else:
                model.add(GRU(32, activation='tanh', return_sequences=False))
            model.add(Dense(1))
            model.compile(optimizer='adamW', loss='mae')
            self.rnn_models[name] = model

    def fit_initial(self, train_df: pl.DataFrame):
        print("Fitting initial models...")
        
        # --- Save Full Train (New) ---
        self.full_train_df = train_df
        if 'date_id' in train_df.columns:
            self.last_train_date = train_df['date_id'].max()
            self.last_date_id = self.last_train_date

        # Slice for training efficiency if needed (logic remains same)
        df = train_df
        
        # Basic Imputation
        df = df.with_columns(
            pl.selectors.numeric().fill_null(
                pl.selectors.numeric().rolling_mean(window_size=5, min_periods=1)
            )
        )

        # Lag Mapping
        lag_mappings = {
            'forward_returns': 'lagged_forward_returns',
            'risk_free_rate': 'lagged_risk_free_rate',
            'market_forward_excess_returns': 'lagged_market_forward_excess_returns',
            'market_forward_return': 'lagged_forward_returns' 
        }
        exprs = []
        for train_col, test_col in lag_mappings.items():
            if train_col in df.columns and test_col not in df.columns:
                exprs.append(pl.col(train_col).shift(1).alias(test_col))
        if exprs:
            df = df.with_columns(exprs)

        # --- 1. Compute SP Index (Continuously) ---
        if 'lagged_forward_returns' in df.columns:
            ret_col = pl.col('lagged_forward_returns')
        else:
            ret_col = pl.lit(0.0)
            
        df = df.with_columns(
            (1 + ret_col.fill_null(0.0)).cum_prod().alias('sp_index')
        )
        self.last_sp_index = df['sp_index'].tail(1).item()

        # --- 2. Define Columns and Pre-Normalize ---
        target_cols = ['market_forward_excess_returns', 'target', 'target_rev', 'forward_returns', 'risk_free_rate', 'market_forward_return']
        exclude_from_raw = ['date_id', 'sp_index'] + target_cols
        
        # History Buffer Cols (Normalized features + sp_index + lags)
        self.raw_cols = [c for c in df.columns if c not in target_cols]
        
        # Identify columns for scaling (Features D1..V9, etc.)
        self.scale_cols = [c for c in df.columns if c not in exclude_from_raw and c in self.raw_cols]
        
        # Cast to Float
        df = df.with_columns(pl.col(self.scale_cols).cast(pl.Float64))
        
        print(f"Fitting Raw Scaler on {len(self.scale_cols)} columns...")
        # Fit Scaler on Raw Data
        train_raw_np = df.select(self.scale_cols).to_numpy()
        train_raw_np = self._sanitize_numpy(train_raw_np)
        self.raw_scaler.fit(train_raw_np)
        
        # Transform Data in-place for Training
        train_norm_np = self.raw_scaler.transform(train_raw_np)
        
        # Replace columns in DF with normalized versions
        df_norm_dict = {col: train_norm_np[:, i] for i, col in enumerate(self.scale_cols)}
        df = df.with_columns([pl.Series(k, v) for k, v in df_norm_dict.items()])

        # Save Normalized History (includes sp_index)
        df = df.with_columns(pl.col("date_id").cast(pl.Int64))
        self.history_df = df.select(self.raw_cols).tail(MAX_HISTORY_LEN)
        

        # --- 3. Feature Generation (on Normalized Data) ---
        df_processed = self._generate_features(df)
        print(df_processed.columns)
        
        # Targets
        if 'market_forward_excess_returns' in df_processed.columns:
            target_col = 'market_forward_excess_returns'
        else:
            target_col = 'target'
        
        y_std = df_processed[target_col].to_numpy()
        #print(y_std)
        
        # Reverse Target Logic
        with np.errstate(divide='ignore', invalid='ignore'):
            rev_target = 1e-3 / (y_std)
        rev_target = np.nan_to_num(rev_target, nan=0.0, posinf=0, neginf=0)
        rev_target = np.where(y_std > 0, rev_target + 0.5, -0.3)
        y_rev = 2 * np.clip(1.5 * rev_target, -0.5, 2.5)

        # Final Feature Selection
        exclude_final = ['date_id', 'sp_index'] + target_cols
        self.feat_cols = [c for c in df_processed.columns if c not in exclude_final]

        # Prepare X (Post-Feature Gen scaling)
        X = df_processed.select(self.feat_cols).to_numpy()
        X = self._sanitize_numpy(X)
        X = self.imputer.fit_transform(X)
        X = self.final_scaler.fit_transform(X) # Standardize again before model

        # --- 4. Training ---
        print("Training Models...")
        self.models['XGB_Std'].fit(X, y_std)
        self.models['XGB_Rev'].fit(X, y_rev)
        self.models['LGB_Std'].fit(X, y_std)
        self.models['LGB_Rev'].fit(X, y_rev)
        self.models['KNN_Std'].fit(X, y_std)
        self.models['KNN_Rev'].fit(X, y_rev)
        self.models['Bagging_Std'].fit(X, y_std)
        self.models['Bagging_Rev'].fit(X, y_rev)

        # --- INSERT THIS WARM-UP BLOCK ---
        print("Warming up Z-Score history...")
        
        # 1. Slice the last 'z_window' (60) rows from the processed training data
        # 'X' is currently the full training set (normalized/imputed)
        if len(X) > self.z_window:
            X_warmup = X[-self.z_window:]
        else:
            X_warmup = X

        # 2. Run predictions on this history to populate self.pred_history
        for name, model in self.models.items():
            if 'Std' in name: # Only needed for Std models where we apply Vol Control
                try:
                    # Predict on the batch
                    warmup_preds = model.predict(X_warmup)
                    
                    # Store in history (convert to list)
                    self.pred_history[name] = list(warmup_preds)
                except Exception as e:
                    print(f"Warmup failed for {name}: {e}")
        # ---------------------------------



        self.fitted = True
        print("Initialization Complete.")


    def predict(self, test_df: pl.DataFrame, revealed_targets: pl.DataFrame = None):
        if not self.fitted:
            return np.zeros(len(test_df))+1

        # 1. Update Buffer / Align Columns
        valid_cols = [c for c in self.raw_cols if c in test_df.columns]
        test_clean = test_df.select(valid_cols)
        
        missing_cols = [c for c in self.raw_cols if c not in test_clean.columns]
        if missing_cols:
            test_clean = test_clean.with_columns([pl.lit(0.0).alias(c) for c in missing_cols])
            test_clean = test_clean.select(self.raw_cols)
        
        for c in test_clean.columns:
            if c != 'date_id':
                test_clean = test_clean.with_columns(pl.col(c).cast(pl.Float64))

        # --- 2. Fix Index Continuity (Smart Look-back) ---
        if 'lagged_forward_returns' in test_clean.columns:
            ret_val = test_clean['lagged_forward_returns'][0]
        elif 'lagged_market_forward_return' in test_clean.columns:
            ret_val = test_clean['lagged_market_forward_return'][0]
        else:
            ret_val = 0.0
            
        if ret_val is None: ret_val = 0.0
        
        curr_date = test_clean['date_id'][0] # Assume 1 row per batch
        
        # Look up previous index to support updates/gaps correctly
        past_ref = self.history_df.filter(pl.col('date_id') < curr_date).tail(1)
        
        if past_ref.height > 0:
            base_idx = past_ref['sp_index'].item()
        else:
            base_idx = self.last_sp_index 

        # Calculate new index for this specific test row
        current_sp_index = base_idx * (1 + ret_val)
        
        # Assign to dataframe
        test_clean = test_clean.with_columns(pl.lit(current_sp_index).alias('sp_index'))

        # --- 3. Pre-Normalize Test Data ---
        # (Compare apples to apples: History is already normalized)
        if self.scale_cols:
            test_raw_np = test_clean.select(self.scale_cols).to_numpy()
            test_raw_np = self._sanitize_numpy(test_raw_np)
            test_norm_np = self.raw_scaler.transform(test_raw_np)
            
            df_norm_dict = {col: test_norm_np[:, i] for i, col in enumerate(self.scale_cols)}
            test_clean = test_clean.with_columns([pl.Series(k, v) for k, v in df_norm_dict.items()])

        # --- 4. Concat with History (Update vs Append vs Gap) ---
        last_hist_date = self.history_df['date_id'].tail(1).item()
        
        combined_raw = None
        
        if curr_date <= last_hist_date:
            # Case A: Update/Correction
            
            # --- SANITY CHECK ---
            # Get the existing row for this date
            existing_row = self.history_df.filter(pl.col('date_id') == curr_date)
            
            is_identical = False
            if existing_row.height > 0:
                # Select only the feature columns (scale_cols) to compare
                # test_clean is now normalized, so we compare directly to history
                cols_to_check = [c for c in self.scale_cols if c in test_clean.columns]
                
                v_old = existing_row.select(cols_to_check).to_numpy()
                v_new = test_clean.select(cols_to_check).to_numpy()
                
                
                # Check for equality (allowing for tiny float precision diffs)
                if np.allclose(v_old, v_new, equal_nan=True, atol=1e-6):
                    is_identical = True
            
            if is_identical:
                # Data is exactly the same, no need to churn memory
                # print(f"Sanity Check Passed: Data for {curr_date} exists and is identical. Skipping update.")
                combined_raw = self.history_df
            else:
                # Data is different (or didn't exist properly), overwrite
                print(f"Data Update Detected: Overwriting history for {curr_date}")
                print(cols_to_check)
                print((v_new - v_old)*1e5)

                history_filtered = self.history_df.filter(pl.col('date_id') != curr_date)
                combined_raw = pl.concat([history_filtered, test_clean]).sort('date_id')
            
        elif curr_date == last_hist_date + 1:
            # Case B: Standard Append
            
            combined_raw = pl.concat([self.history_df, test_clean], how="vertical")
            self.last_date_id = curr_date
            
        else:
            # Case C: Gap detected -> Interpolate
            start_row = self.history_df.tail(1)
            end_row = test_clean
            
            gap_dates = list(range(last_hist_date + 1, curr_date))
            
            if len(gap_dates) > 0:
                print('histroy extended by interpolation')
                gap_data = {col: [None]*len(gap_dates) for col in self.history_df.columns}
                gap_data['date_id'] = gap_dates
                gap_df = pl.DataFrame(gap_data, schema=self.history_df.schema)
                
                interp_base = pl.concat([start_row, gap_df, end_row], how="vertical")
                interp_filled = interp_base.interpolate()
                
                to_append = interp_filled.slice(1, len(gap_dates) + 1)
                
                to_append = to_append.with_columns(pl.col("date_id").cast(pl.Int64))
                print(to_append)
                combined_raw = pl.concat([self.history_df, to_append], how="vertical")
            else:
                combined_raw = pl.concat([self.history_df, test_clean], how="vertical")
        
        # 5. Maintain Window Size & Update State
        if combined_raw.height > MAX_HISTORY_LEN:
            self.history_df = combined_raw.tail(MAX_HISTORY_LEN)
        else:
            self.history_df = combined_raw
            
        self.last_sp_index = self.history_df['sp_index'].tail(1).item()
        
        # 6. Generate Features (On Updated History)
        # We must regenerate because even if data is identical, 
        # rolling features rely on the sequence context
        processed_full = self._generate_features(self.history_df)
        
        # 7. Extract Specific Row for Prediction
        current_features_df = processed_full.filter(pl.col('date_id') == curr_date)
        
        if current_features_df.height == 0:
             current_features_df = processed_full.tail(1)
             
        n_test = len(current_features_df)

        # 8. Prepare X
        X_raw = current_features_df.select([
            pl.col(c) if c in current_features_df.columns else pl.lit(0.0).alias(c) 
            for c in self.feat_cols
        ]).to_numpy()
        
        X_raw = self._sanitize_numpy(X_raw)
        X = self.final_scaler.transform(self.imputer.transform(X_raw))
        X = self._sanitize_numpy(X)

        # --- INSERT/MODIFY THIS SECTION ---
        # A. Calculate Volatility Scalar for this step
        current_vol_scalar = self._get_vol_scalar()

        # B. Predictions & Vol Control
        preds = {}
        
        # 9. Predictions
        preds = {}
        def to_signal(p, is_std=True):
            if is_std: return np.clip(p * 300.0 + 0.8, 0.0, 2.0)
            return np.clip(p, 0.0, 2.0)
        

        for name, model in self.models.items():
            try:
                raw = model.predict(X)
                preds[name] = to_signal(raw, is_std=('Std' in name))
                # [NEW] Volatility Control Add-on
                # For 'Std' models, generate a Vol-Controlled version
                if 'Std' in name:
                    # raw[0] because predict returns array and we handle 1 date
                    # print(current_vol_scalar)
                    vol_sig = self._apply_vol_control(name, raw[0], current_vol_scalar)
                    preds[f"{name}_Vol"] = np.array([vol_sig] * n_test)
            except Exception:
                preds[name] = np.ones(n_test)*0.8

        for ps in ['strat_rsi_mr', 'strat_rsi_mom', 'strat_ma_cross', 'strat_ma_dist', 'strat_bb_break', 'strat_tsmom']:
            if ps in current_features_df.columns:
                preds[ps] = current_features_df[ps].to_numpy()
            else:
                preds[ps] = np.ones(n_test)

        # 10. Ensemble
        all_preds = np.column_stack(list(preds.values()))
        all_preds = np.nan_to_num(all_preds, nan=1.0)

        
        ens_names = ['KNN_Rev']
        ens = [preds[c] for c in ens_names]
        print(ens)
        return np.mean(ens)




In [4]:

#KNN_Rev 0.67, KNN_Std_Vol 0.78, ensemble 0.8
def generate_features(df: pl.DataFrame) -> pl.DataFrame:
  """Generates new features from the base polars dataframe.
    
      Available Feature Categories:
      - D* (Dummy/Binary features): 9 columns (D1-D9)
      - E* (Macro Economic features): 20 columns (E1-E20)
      - I* (Interest Rate features): 9 columns (I1-I9)
      - M* (Market Dynamics/Technical features): 18 columns (M1-M18)
      - P* (Price/Valuation features): 13 columns (P1-P13)
      - S* (Sentiment features): 12 columns (S1-S12)
      - V* (Volatility features): 13 columns (V1-V13)
    
        Top 20 Most Positively Correlated Features with Target:
        ['V13', 'M1', 'S5', 'D1', 'D2', 'M2', 'V10', 'V7', 'S12', 'S6', 'M17', 'D8', 'E19', 'D4', 'D6', 'V9', 'M3', 'D7', 'E9', 'V6']
    
        Top 19 Most Negatively Correlated Features with Target:
        ['M4', 'S2', 'P8', 'E7', 'E11', 'E12', 'M12', 'I2', 'P7', 'P5', 'P10', 'P12', 'M8', 'S3', 'S7', 'P11', 'P3', 'E13', 'I1']
    
        Do not look into future, e.g. do not use negative lags or backward fill.
  """
  new_features = pl.DataFrame({
    # --- Retained Features from v0 ---
    'feat_M1_roll_median_10': df['M1'].rolling_median(window_size=10),
    'feat_V13_roll_median_10': df['V13'].rolling_median(window_size=10),
    'feat_S5_roll_median_5': df['S5'].rolling_median(window_size=5),
    'feat_D1_roll_mean_10': df['D1'].rolling_mean(window_size=10),
    'feat_D2_roll_std_5': df['D2'].rolling_std(window_size=5),
    'feat_D4_roll_mean_20': df['D4'].rolling_mean(window_size=20),
    'feat_D6_roll_mean_10': df['D6'].rolling_mean(window_size=10),
    'feat_D7_roll_std_5': df['D7'].rolling_std(window_size=5),
    'feat_M1_roll_10_vs_P1_roll_10_diff': df['M1'].rolling_mean(window_size=10) - df['P1'].rolling_mean(window_size=10),
    'feat_V13_roll_10_vs_M1_roll_10_diff': df['V13'].rolling_mean(window_size=10) - df['M1'].rolling_mean(window_size=10),
    'feat_S5_roll_5_vs_D1_roll_5_diff': df['S5'].rolling_mean(window_size=5) - df['D1'].rolling_mean(window_size=5),
    'feat_S12_lag_3': df['S12'].shift(3),
    'feat_V10_lag_2': df['V10'].shift(2),
    'feat_E19_lag_1': df['E19'].shift(1),
    'feat_M1_roll_max_5': df['M1'].rolling_max(window_size=5),
    'feat_M1_roll_min_5': df['M1'].rolling_min(window_size=5),
    'feat_V13_roll_max_10': df['V13'].rolling_max(window_size=10),
    'feat_V13_roll_min_10': df['V13'].rolling_min(window_size=10),
    'feat_M1_diff_1_1': df['M1'] - df['M1'].shift(1),
    'feat_V13_diff_1_1': df['V13'] - df['V13'].shift(1),
    'feat_S5_diff_1_1': df['S5'] - df['S5'].shift(1),
    'feat_M1_roll_std_10': df['M1'].rolling_std(window_size=10),
    'feat_V13_roll_std_10': df['V13'].rolling_std(window_size=10),
    'feat_S5_roll_std_5': df['S5'].rolling_std(window_size=5),
    'feat_M4_lag_1': df['M4'].shift(1),
    'feat_P8_roll_mean_5': df['P8'].rolling_mean(window_size=5),
    'feat_E7_roll_std_10': df['E7'].rolling_std(window_size=10),
    'feat_M1_mul_V13_ratio_10': df['M1'].rolling_mean(10) / df['V13'].rolling_mean(10),
    'feat_D8_roll_mean_10': df['D8'].rolling_mean(window_size=10),
    'feat_E9_roll_std_10': df['E9'].rolling_std(window_size=10),
    'feat_V6_roll_diff_5': df['V6'].rolling_mean(window_size=5) - df['V6'].shift(5),
    'feat_V7_roll_mean_5': df['V7'].rolling_mean(window_size=5),
    'feat_V9_roll_mean_5': df['V9'].rolling_mean(window_size=5),
    'feat_M3_roll_std_10': df['M3'].rolling_std(window_size=10),
    'feat_M1_div_V13_10': df['M1'].rolling_mean(10) / df['V13'].rolling_mean(10),
    'feat_S5_div_D1_5': df['S5'].rolling_mean(5) / df['D1'].rolling_mean(5),
    'feat_P7_roll_std_5': df['P7'].rolling_std(window_size=5),
    'feat_I2_lag_2': df['I2'].shift(2),
    'feat_E11_roll_mean_10': df['E11'].rolling_mean(window_size=10),
    'feat_D8_diff_1': df['D8'] - df['D8'].shift(1),
    'feat_V7_diff_1': df['V7'] - df['V7'].shift(1),
    'feat_S12_roll_mean_5': df['S12'].rolling_mean(window_size=5),
    'feat_V10_roll_std_5': df['V10'].rolling_std(window_size=5),
    'feat_V10_roll_min_5': df['V10'].rolling_min(window_size=5),
    'feat_S12_roll_std_5': df['S12'].rolling_std(window_size=5),
    # Features added in v1 (V7, M3, E19, D8 momentum/stats, M1/S5 ratio)
    'feat_V7_roll_std_5': df['V7'].rolling_std(window_size=5),
    'feat_V7_roll_max_5': df['V7'].rolling_max(window_size=5),
    'feat_M3_roll_mean_5': df['M3'].rolling_mean(window_size=5),
    'feat_M3_roll_std_5': df['M3'].rolling_std(window_size=5),
    'feat_E19_roll_mean_5': df['E19'].rolling_mean(window_size=5),
    'feat_E19_diff_1': df['E19'] - df['E19'].shift(1),
    'feat_D8_roll_std_10': df['D8'].rolling_std(window_size=10),
    'feat_M1_div_S5_10': df['M1'].rolling_mean(10) / df['S5'].rolling_mean(10),
  })
  # --- New additions for v1 ---
  # P8 (Neg Corr)
  new_features = new_features.with_columns(
    (df['P8'].rolling_std(window_size=10).alias('feat_P8_roll_std_10')),
    (df['P8'].shift(5).alias('feat_P8_lag_5'))
  )
  # P7 (Neg Corr)
  new_features = new_features.with_columns(
    (df['P7'].rolling_mean(window_size=10).alias('feat_P7_roll_mean_10')),
    (df['P7'].shift(1).alias('feat_P7_lag_1'))
  )
  # I2 (Neg Corr)
  new_features = new_features.with_columns(
    (df['I2'].rolling_mean(window_size=5).alias('feat_I2_roll_mean_5')),
    (df['I2'].diff().alias('feat_I2_diff_1'))
  )
  # E11 (Neg Corr)
  new_features = new_features.with_columns(
    (df['E11'].rolling_std(window_size=5).alias('feat_E11_roll_std_5')),
    (df['E11'].shift(5).alias('feat_E11_lag_5'))
  )
  # Adding some cross-feature momentum (M1/V13 momentum vs M1/V13 current state)
  new_features = new_features.with_columns(
    ((df['M1'].rolling_mean(10) - df['M1'].shift(5)).alias('feat_M1_roll_10_momentum_5')),
    ((df['V13'].rolling_mean(10) - df['V13'].shift(5)).alias('feat_V13_roll_10_momentum_5')),
  )
  # Adding exponential moving averages
  new_features = new_features.with_columns(
    (df['M1'].ewm_mean(alpha=0.2).alias('feat_M1_ewm_0.2')),
    (df['V13'].ewm_mean(alpha=0.2).alias('feat_V13_ewm_0.2'))
  )
  # New additions: Adding rolling skewness for key features
  new_features = new_features.with_columns(
    (df['M1'].rolling_skew(window_size=10).alias('feat_M1_roll_skew_10')),
    (df['V13'].rolling_skew(window_size=10).alias('feat_V13_roll_skew_10')),
    (df['S5'].rolling_skew(window_size=5).alias('feat_S5_roll_skew_5'))
  )
  # New additions: Adding rolling kurtosis for key features
  import scipy.stats as st
#   new_features = new_features.with_columns(
#     (df['M1'].rolling_kurtosis(window_size=10).alias('feat_M1_roll_kurt_10')),
#     (df['V13'].rolling_kurtosis(window_size=10).alias('feat_V13_roll_kurt_10')),
#     (df['S5'].rolling_kurtosis(window_size=5).alias('feat_S5_roll_kurt_5'))
#   )
#   # New additions: Adding time series rank features
#   new_features = new_features.with_columns(
#     (df['M1'].rolling_rank(window_size=10).alias('feat_M1_roll_rank_10')),
#     (df['V13'].rolling_rank(window_size=10).alias('feat_V13_roll_rank_10')),
#     (df['S5'].rolling_rank(window_size=5).alias('feat_S5_roll_rank_5'))
#  )

  # New additions: Adding rolling kurtosis for key features (using scipy.stats workaround)
  new_features = new_features.with_columns(
    (df['M1'].rolling_map(lambda x: st.kurtosis(x, fisher=True, bias=False), window_size=10).alias('feat_M1_roll_kurt_10')),
    (df['V13'].rolling_map(lambda x: st.kurtosis(x, fisher=True, bias=False), window_size=10).alias('feat_V13_roll_kurt_10')),
    (df['S5'].rolling_map(lambda x: st.kurtosis(x, fisher=True, bias=False), window_size=5).alias('feat_S5_roll_kurt_5'))
  )

# New additions: Adding time series rank features (using scipy.stats workaround)
# Note: st.rankdata returns the ranks of the window; we take the last one [-1] for the current time step.
  new_features = new_features.with_columns(
    (df['M1'].rolling_map(lambda x: st.rankdata(x)[-1], window_size=10).alias('feat_M1_roll_rank_10')),
    (df['V13'].rolling_map(lambda x: st.rankdata(x)[-1], window_size=10).alias('feat_V13_roll_rank_10')),
    (df['S5'].rolling_map(lambda x: st.rankdata(x)[-1], window_size=5).alias('feat_S5_roll_rank_5'))
  )
  return new_features.with_columns(pl.all().forward_fill())






class OnlineStrategy:

    def __init__(self):
        self.history_df = pl.DataFrame()
        self.fitted = False
        self.feat_cols = [] 
        self.raw_cols = [] 
        
        # --- State for Continuity ---
        self.last_date_id = 0
        self.last_sp_index = 1.0
        self.raw_scaler = StandardScaler()
        self.scale_cols = [] # Columns that need pre-normalization
        
        # --- Storage for Retraining ---
        self.full_train_df = None
        self.last_train_date = -1

        # Volatility Control State
        self.pred_history = {}  # Format: {'Model_Name': [val1, val2, ...]}
        self.vol_target = 20
        self.vol_window = 20
        self.z_window = 60
        self.leverage_cap = 2.0
        # -------------------------

        # --- Models ---\n
        self.models = {
            'XGB_Std': xgb.XGBRegressor(n_estimators=2, max_depth=5, n_jobs=-1, random_state=42),
            'XGB_Rev': xgb.XGBRegressor(n_estimators=2, max_depth=5, n_jobs=-1, random_state=42),
            'LGB_Std': lgb.LGBMRegressor(n_estimators=2, max_depth=4, n_jobs=-1, random_state=42, verbose=-1),
            'LGB_Rev': lgb.LGBMRegressor(n_estimators=2, max_depth=4, n_jobs=-1, random_state=42, verbose=-1),
            'KNN_Std': KNeighborsRegressor(n_neighbors=40, n_jobs=-1),
            'KNN_Rev': KNeighborsRegressor(n_neighbors=40, n_jobs=-1),
            'Bagging_Std': BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=5), n_estimators=10, random_state=42, n_jobs=-1),
            'Bagging_Rev': BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=5), n_estimators=10, random_state=42, n_jobs=-1),
        }
        self.rnn_models = {}
        # self.rnn_names = ['LSTM_Std', 'LSTM_Rev', 'GRU_Std', 'GRU_Rev']
        self.rnn_names = [] # Disabled for speed in this snippet, re-enable if needed
        
        # Preprocessing for Final X (post-feature-gen)
        self.imputer = SimpleImputer(strategy='constant', fill_value=0)
        self.final_scaler = StandardScaler()


    # --- INSERT THESE NEW METHODS ---
    def _get_vol_scalar(self):
        """Calculates (Target_Vol / Current_Vol) based on history."""
        # Check for 'lagged_forward_returns' or 'lagged_market_forward_return'
        col_name = None
        if 'lagged_forward_returns' in self.history_df.columns:
            col_name = 'lagged_forward_returns'
        elif 'lagged_market_forward_return' in self.history_df.columns:
            col_name = 'lagged_market_forward_return'
            
        if col_name and self.history_df.height >= 5:
            # Get last N days
            returns = self.history_df[col_name].tail(self.vol_window).to_numpy()
            
            # FIX: Use keyword argument for 'nan' to avoid passing 0.0 as 'copy'
            returns = np.nan_to_num(returns, nan=0.0) 
            
            # Calculate Daily Vol
            daily_vol = np.std(returns)
            if daily_vol < 1e-6: daily_vol = 0.005 # Avoid div by zero
            
            # Annualize target to daily
            daily_target = self.vol_target / np.sqrt(TRADING_DAYS_PER_YR)
            
            scalar = daily_target / daily_vol
            return np.clip(scalar, 0, self.leverage_cap)
        return 1.0

    def _apply_vol_control(self, name, raw_pred, vol_scalar):
        """Online implementation of: Position = Vol_Scalar * Tanh(Z_Score(Pred))"""
        # 1. Update History
        if name not in self.pred_history:
            self.pred_history[name] = []
        self.pred_history[name].append(raw_pred)
        
        if len(self.pred_history[name]) > self.z_window:
            self.pred_history[name].pop(0)
            
        # 2. Calculate Stats (Z-Score)
        history_arr = np.array(self.pred_history[name])
        if len(history_arr) < 5:
            mu = np.mean(history_arr)
            sigma = np.std(history_arr) + 1e-6
        else:
            mu = np.mean(history_arr)
            sigma = np.std(history_arr) + 1e-6
            
        z_score = (raw_pred - mu) / sigma
        
        # 3. Tanh Activation & Scalar
        return np.clip(vol_scalar * np.tanh(z_score), 0.0, 2.0)
    # --------------------------------

    def _generate_features(self, df: pl.DataFrame) -> pl.DataFrame:
        """
        Generates complex features. 
        CRITICAL: Expects 'sp_index' to ALREADY exist and be continuous.
        CRITICAL: Expects raw feature columns (M1, P1, etc.) to ALREADY be normalized.
        """
        if 'date_id' in df.columns:
            df = df.sort('date_id')

        # 1. Technical Indicators (rely on sp_index)
        def calculate_rsi_expr(price_col, period=14):
            delta = price_col.diff()
            up = delta.clip(lower_bound=0)
            down = delta.clip(upper_bound=0).abs()
            roll_up = up.rolling_mean(period)
            roll_down = down.rolling_mean(period)
            rs = roll_up / (roll_down + 1e-9)
            return 100.0 - (100.0 / (1.0 + rs))

        df = df.with_columns([
            calculate_rsi_expr(pl.col('sp_index'), 14).alias('ind_rsi_14'),
            calculate_rsi_expr(pl.col('sp_index'), 42).alias('ind_rsi_42'),
            (pl.col('sp_index').rolling_mean(5) / (pl.col('sp_index') + 1e-9)).alias('ind_sma_5'),
            (pl.col('sp_index').rolling_mean(30) / (pl.col('sp_index') + 1e-9)).alias('ind_sma_30'),
            (pl.col('sp_index').rolling_mean(200) / (pl.col('sp_index') + 1e-9)).alias('ind_sma_200'),
            (pl.col('sp_index').rolling_std(50) / (pl.col('sp_index') + 1e-9)).alias('ind_std_10'),
            (pl.col('sp_index').rolling_std(50) / (pl.col('sp_index') + 1e-9)).alias('ind_std_50'),
            (pl.col('sp_index').shift(TRADING_DAYS_PER_YR) / (pl.col('sp_index') + 1e-9)).alias('ind_price_1y_ago'),
            (pl.col('sp_index').rolling_mean(20) / (pl.col('sp_index') + 1e-9)).alias('ind_bb_mid'),
            (pl.col('sp_index').rolling_std(20) / (pl.col('sp_index') + 1e-9)).alias('ind_bb_std'),
        ])
        
        df = df.with_columns([
            (pl.col('ind_bb_mid') + 2 * pl.col('ind_bb_std')).alias('ind_bb_upper'),
            (pl.col('ind_bb_mid') - 2 * pl.col('ind_bb_std')).alias('ind_bb_lower')
        ])

        # 2. Strategies
        sig_rsi_mr = pl.when(pl.col('ind_rsi_14') < 30).then(1.5).when(pl.col('ind_rsi_14') > 70).then(0.5).otherwise(0.8)
        sig_rsi_mom = pl.when(pl.col('ind_rsi_14') > 50).then(1.5).otherwise(0.6)
        sig_ma_cross = pl.when(pl.col('ind_sma_30') > pl.col('ind_sma_200')).then(1.5).otherwise(0.5)
        
        z_score = (1.0 - pl.col('ind_sma_30')) / (pl.col('ind_std_50') + 1e-9)
        sig_ma_dist = pl.when(z_score < -2.0).then(1.5).when(z_score > 2.0).then(0.5).otherwise(1.0)
        
        sig_bb_break = pl.when(1.0 > pl.col('ind_bb_upper')).then(1.5).when(1.0 < pl.col('ind_bb_lower')).then(0.5).otherwise(0.8)
        sig_tsmom = pl.when(1.0 > pl.col('ind_price_1y_ago')).then(1.5).otherwise(0.6)

        df = df.with_columns([
            sig_rsi_mr.fill_null(1.0).alias('strat_rsi_mr'),
            sig_rsi_mom.fill_null(1.0).alias('strat_rsi_mom'),
            sig_ma_cross.fill_null(1.0).alias('strat_ma_cross'),
            sig_ma_dist.fill_null(1.0).alias('strat_ma_dist'),
            sig_bb_break.fill_null(1.0).alias('strat_bb_break'),
            sig_tsmom.fill_null(1.0).alias('strat_tsmom'),
        ])

        # 3. Complex Features (Uses Pre-Normalized Data)
        # def safe_col(name): return pl.col(name) if name in df.columns else pl.lit(0.0)

        # new_features = df.with_columns([
        #     (safe_col('M1') * safe_col('V1')).alias('feat_M1_x_V1'),
        #     (safe_col('P1') + safe_col('E1')).alias('feat_P1_add_E1'),
        #     (safe_col('S1') - safe_col('I1')).alias('feat_S1_sub_I1'),
        #     safe_col('V2').rolling_mean(5).alias('feat_V2_roll_mean_5'),
        #     safe_col('V1').rolling_std(5).alias('feat_V1_roll_std_5'),
        #     safe_col('M1').rolling_mean(20).alias('feat_M1_roll_mean_20'),
        #     safe_col('M3').rolling_std(20).alias('feat_M3_roll_std_20'),
        #     safe_col('P1').rolling_max(10).alias('feat_P1_roll_max_10'),
        #     safe_col('P1').rolling_min(10).alias('feat_P1_roll_min_10'),
        #     (safe_col('M1').rolling_mean(5) - safe_col('M1').rolling_mean(20)).alias('feat_roll_diff_M1_5_20'),
        # ])
        
        # Cleanup
        add_new_features = generate_features(df)
        new_features = pl.concat([df, add_new_features], how='horizontal')
        new_features = new_features.fill_null(0).fill_nan(0)
        return new_features

    def _sanitize_numpy(self, X):
        return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    def _build_rnn(self, input_shape):

        for name in self.rnn_names:
            model = Sequential()
            model.add(Input(shape=input_shape))
            if 'LSTM' in name:
                model.add(LSTM(32, activation='tanh', return_sequences=False))
            else:
                model.add(GRU(32, activation='tanh', return_sequences=False))
            model.add(Dense(1))
            model.compile(optimizer='adamW', loss='mae')
            self.rnn_models[name] = model

    def fit_initial(self, train_df: pl.DataFrame):
        print("Fitting initial models...")
        
        # --- Save Full Train (New) ---
        self.full_train_df = train_df
        if 'date_id' in train_df.columns:
            self.last_train_date = train_df['date_id'].max()
            self.last_date_id = self.last_train_date

        # Slice for training efficiency if needed (logic remains same)
        df = train_df
        
        # Basic Imputation
        df = df.with_columns(
            pl.selectors.numeric().fill_null(
                pl.selectors.numeric().rolling_mean(window_size=5, min_periods=1)
            )
        )

        # Lag Mapping
        lag_mappings = {
            'forward_returns': 'lagged_forward_returns',
            'risk_free_rate': 'lagged_risk_free_rate',
            'market_forward_excess_returns': 'lagged_market_forward_excess_returns',
            'market_forward_return': 'lagged_forward_returns' 
        }
        exprs = []
        for train_col, test_col in lag_mappings.items():
            if train_col in df.columns and test_col not in df.columns:
                exprs.append(pl.col(train_col).shift(1).alias(test_col))
        if exprs:
            df = df.with_columns(exprs)

        # --- 1. Compute SP Index (Continuously) ---
        if 'lagged_forward_returns' in df.columns:
            ret_col = pl.col('lagged_forward_returns')
        else:
            ret_col = pl.lit(0.0)
            
        df = df.with_columns(
            (1 + ret_col.fill_null(0.0)).cum_prod().alias('sp_index')
        )
        self.last_sp_index = df['sp_index'].tail(1).item()

        # --- 2. Define Columns and Pre-Normalize ---
        target_cols = ['market_forward_excess_returns', 'target', 'target_rev', 'forward_returns', 'risk_free_rate', 'market_forward_return']
        exclude_from_raw = ['date_id', 'sp_index'] + target_cols
        
        # History Buffer Cols (Normalized features + sp_index + lags)
        self.raw_cols = [c for c in df.columns if c not in target_cols]
        
        # Identify columns for scaling (Features D1..V9, etc.)
        self.scale_cols = [c for c in df.columns if c not in exclude_from_raw and c in self.raw_cols]
        
        # Cast to Float
        df = df.with_columns(pl.col(self.scale_cols).cast(pl.Float64))
        
        print(f"Fitting Raw Scaler on {len(self.scale_cols)} columns...")
        # Fit Scaler on Raw Data
        train_raw_np = df.select(self.scale_cols).to_numpy()
        train_raw_np = self._sanitize_numpy(train_raw_np)
        self.raw_scaler.fit(train_raw_np)
        
        # Transform Data in-place for Training
        train_norm_np = self.raw_scaler.transform(train_raw_np)
        
        # Replace columns in DF with normalized versions
        df_norm_dict = {col: train_norm_np[:, i] for i, col in enumerate(self.scale_cols)}
        df = df.with_columns([pl.Series(k, v) for k, v in df_norm_dict.items()])

        # Save Normalized History (includes sp_index)
        df = df.with_columns(pl.col("date_id").cast(pl.Int64))
        self.history_df = df.select(self.raw_cols).tail(MAX_HISTORY_LEN)
        

        # --- 3. Feature Generation (on Normalized Data) ---
        df_processed = self._generate_features(df)
        print(df_processed.columns)
        
        # Targets
        if 'market_forward_excess_returns' in df_processed.columns:
            target_col = 'market_forward_excess_returns'
        else:
            target_col = 'target'
        
        y_std = df_processed[target_col].to_numpy()
        #print(y_std)
        
        # Reverse Target Logic
        with np.errstate(divide='ignore', invalid='ignore'):
            rev_target = 1e-3 / (y_std)
        rev_target = np.nan_to_num(rev_target, nan=0.0, posinf=0, neginf=0)
        rev_target = np.where(y_std > 0, rev_target + 0.5, -0.3)
        y_rev = 2 * np.clip(1.5 * rev_target, -0.5, 2.5)

        # Final Feature Selection
        exclude_final = ['date_id', 'sp_index'] + target_cols
        self.feat_cols = [c for c in df_processed.columns if c not in exclude_final]

        # Prepare X (Post-Feature Gen scaling)
        X = df_processed.select(self.feat_cols).to_numpy()
        X = self._sanitize_numpy(X)
        X = self.imputer.fit_transform(X)
        X = self.final_scaler.fit_transform(X) # Standardize again before model

        # --- 4. Training ---
        print("Training Models...")
        self.models['XGB_Std'].fit(X, y_std)
        self.models['XGB_Rev'].fit(X, y_rev)
        self.models['LGB_Std'].fit(X, y_std)
        self.models['LGB_Rev'].fit(X, y_rev)
        self.models['KNN_Std'].fit(X, y_std)
        self.models['KNN_Rev'].fit(X, y_rev)
        self.models['Bagging_Std'].fit(X, y_std)
        self.models['Bagging_Rev'].fit(X, y_rev)

        # --- INSERT THIS WARM-UP BLOCK ---
        print("Warming up Z-Score history...")
        
        # 1. Slice the last 'z_window' (60) rows from the processed training data
        # 'X' is currently the full training set (normalized/imputed)
        if len(X) > self.z_window:
            X_warmup = X[-self.z_window:]
        else:
            X_warmup = X

        # 2. Run predictions on this history to populate self.pred_history
        for name, model in self.models.items():
            if 'Std' in name: # Only needed for Std models where we apply Vol Control
                try:
                    # Predict on the batch
                    warmup_preds = model.predict(X_warmup)
                    
                    # Store in history (convert to list)
                    self.pred_history[name] = list(warmup_preds)
                except Exception as e:
                    print(f"Warmup failed for {name}: {e}")
        # ---------------------------------



        self.fitted = True
        print("Initialization Complete.")


    def predict(self, test_df: pl.DataFrame, revealed_targets: pl.DataFrame = None):
        if not self.fitted:
            return np.zeros(len(test_df))+1

        # 1. Update Buffer / Align Columns
        valid_cols = [c for c in self.raw_cols if c in test_df.columns]
        test_clean = test_df.select(valid_cols)
        
        missing_cols = [c for c in self.raw_cols if c not in test_clean.columns]
        if missing_cols:
            test_clean = test_clean.with_columns([pl.lit(0.0).alias(c) for c in missing_cols])
            test_clean = test_clean.select(self.raw_cols)
        
        for c in test_clean.columns:
            if c != 'date_id':
                test_clean = test_clean.with_columns(pl.col(c).cast(pl.Float64))

        # --- 2. Fix Index Continuity (Smart Look-back) ---
        if 'lagged_forward_returns' in test_clean.columns:
            ret_val = test_clean['lagged_forward_returns'][0]
        elif 'lagged_market_forward_return' in test_clean.columns:
            ret_val = test_clean['lagged_market_forward_return'][0]
        else:
            ret_val = 0.0
            
        if ret_val is None: ret_val = 0.0
        
        curr_date = test_clean['date_id'][0] # Assume 1 row per batch
        
        # Look up previous index to support updates/gaps correctly
        past_ref = self.history_df.filter(pl.col('date_id') < curr_date).tail(1)
        
        if past_ref.height > 0:
            base_idx = past_ref['sp_index'].item()
        else:
            base_idx = self.last_sp_index 

        # Calculate new index for this specific test row
        current_sp_index = base_idx * (1 + ret_val)
        
        # Assign to dataframe
        test_clean = test_clean.with_columns(pl.lit(current_sp_index).alias('sp_index'))

        # --- 3. Pre-Normalize Test Data ---
        # (Compare apples to apples: History is already normalized)
        if self.scale_cols:
            test_raw_np = test_clean.select(self.scale_cols).to_numpy()
            test_raw_np = self._sanitize_numpy(test_raw_np)
            test_norm_np = self.raw_scaler.transform(test_raw_np)
            
            df_norm_dict = {col: test_norm_np[:, i] for i, col in enumerate(self.scale_cols)}
            test_clean = test_clean.with_columns([pl.Series(k, v) for k, v in df_norm_dict.items()])

        # --- 4. Concat with History (Update vs Append vs Gap) ---
        last_hist_date = self.history_df['date_id'].tail(1).item()
        
        combined_raw = None
        
        if curr_date <= last_hist_date:
            # Case A: Update/Correction
            
            # --- SANITY CHECK ---
            # Get the existing row for this date
            existing_row = self.history_df.filter(pl.col('date_id') == curr_date)
            
            is_identical = False
            if existing_row.height > 0:
                # Select only the feature columns (scale_cols) to compare
                # test_clean is now normalized, so we compare directly to history
                cols_to_check = [c for c in self.scale_cols if c in test_clean.columns]
                
                v_old = existing_row.select(cols_to_check).to_numpy()
                v_new = test_clean.select(cols_to_check).to_numpy()
                
                
                # Check for equality (allowing for tiny float precision diffs)
                if np.allclose(v_old, v_new, equal_nan=True, atol=1e-6):
                    is_identical = True
            
            if is_identical:
                # Data is exactly the same, no need to churn memory
                # print(f"Sanity Check Passed: Data for {curr_date} exists and is identical. Skipping update.")
                combined_raw = self.history_df
            else:
                # Data is different (or didn't exist properly), overwrite
                print(f"Data Update Detected: Overwriting history for {curr_date}")
                print(cols_to_check)
                print((v_new - v_old)*1e5)

                history_filtered = self.history_df.filter(pl.col('date_id') != curr_date)
                combined_raw = pl.concat([history_filtered, test_clean]).sort('date_id')
            
        elif curr_date == last_hist_date + 1:
            # Case B: Standard Append
            
            combined_raw = pl.concat([self.history_df, test_clean], how="vertical")
            self.last_date_id = curr_date
            
        else:
            # Case C: Gap detected -> Interpolate
            start_row = self.history_df.tail(1)
            end_row = test_clean
            
            gap_dates = list(range(last_hist_date + 1, curr_date))
            
            if len(gap_dates) > 0:
                print('histroy extended by interpolation')
                gap_data = {col: [None]*len(gap_dates) for col in self.history_df.columns}
                gap_data['date_id'] = gap_dates
                gap_df = pl.DataFrame(gap_data, schema=self.history_df.schema)
                
                interp_base = pl.concat([start_row, gap_df, end_row], how="vertical")
                interp_filled = interp_base.interpolate()
                
                to_append = interp_filled.slice(1, len(gap_dates) + 1)
                
                to_append = to_append.with_columns(pl.col("date_id").cast(pl.Int64))
                print(to_append)
                combined_raw = pl.concat([self.history_df, to_append], how="vertical")
            else:
                combined_raw = pl.concat([self.history_df, test_clean], how="vertical")
        
        # 5. Maintain Window Size & Update State
        if combined_raw.height > MAX_HISTORY_LEN:
            self.history_df = combined_raw.tail(MAX_HISTORY_LEN)
        else:
            self.history_df = combined_raw
            
        self.last_sp_index = self.history_df['sp_index'].tail(1).item()
        
        # 6. Generate Features (On Updated History)
        # We must regenerate because even if data is identical, 
        # rolling features rely on the sequence context
        processed_full = self._generate_features(self.history_df)
        
        # 7. Extract Specific Row for Prediction
        current_features_df = processed_full.filter(pl.col('date_id') == curr_date)
        
        if current_features_df.height == 0:
             current_features_df = processed_full.tail(1)
             
        n_test = len(current_features_df)

        # 8. Prepare X
        X_raw = current_features_df.select([
            pl.col(c) if c in current_features_df.columns else pl.lit(0.0).alias(c) 
            for c in self.feat_cols
        ]).to_numpy()
        
        X_raw = self._sanitize_numpy(X_raw)
        X = self.final_scaler.transform(self.imputer.transform(X_raw))
        X = self._sanitize_numpy(X)

        # --- INSERT/MODIFY THIS SECTION ---
        # A. Calculate Volatility Scalar for this step
        current_vol_scalar = self._get_vol_scalar()

        # B. Predictions & Vol Control
        preds = {}
        
        # 9. Predictions
        preds = {}
        def to_signal(p, is_std=True):
            if is_std: return np.clip(p * 300.0 + 0.8, 0.0, 2.0)
            return np.clip(p, 0.0, 2.0)
        

        for name, model in self.models.items():
            try:
                raw = model.predict(X)
                preds[name] = to_signal(raw, is_std=('Std' in name))
                # [NEW] Volatility Control Add-on
                # For 'Std' models, generate a Vol-Controlled version
                if 'Std' in name:
                    # raw[0] because predict returns array and we handle 1 date
                    # print(current_vol_scalar)
                    vol_sig = self._apply_vol_control(name, raw[0], current_vol_scalar)
                    preds[f"{name}_Vol"] = np.array([vol_sig] * n_test)
            except Exception:
                preds[name] = np.ones(n_test)*0.8

        for ps in ['strat_rsi_mr', 'strat_rsi_mom', 'strat_ma_cross', 'strat_ma_dist', 'strat_bb_break', 'strat_tsmom']:
            if ps in current_features_df.columns:
                preds[ps] = current_features_df[ps].to_numpy()
            else:
                preds[ps] = np.ones(n_test)

        # 10. Ensemble
        all_preds = np.column_stack(list(preds.values()))
        all_preds = np.nan_to_num(all_preds, nan=1.0)

        
        ens_names = ['KNN_Rev','KNN_Std_Vol']
        ens = [preds[c] for c in ens_names]
        print(ens)
        return np.mean(ens)



In [5]:


# Usage
strategy = OnlineStrategy()
#strategy.fit_initial(pl.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv'))
Ndiscard = 6000
df = pl.read_csv('./kaggle/train.csv')
strategy.fit_initial(df.slice(Ndiscard,len(df) - Ndiscard))

allpreds = []

def predict(test_df, revealed_targets=None):
    #print(test_df.head())
    if isinstance(test_df, pd.DataFrame):
        test_df = pl.from_pandas(test_df)
    dfpred =  strategy.predict(test_df, revealed_targets)
    #print(dfpred, strategy.last_date_id)
    allpreds.append(dfpred.item())
    return dfpred.item()


Fitting initial models...
Fitting Raw Scaler on 97 columns...
['date_id', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'E1', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'M1', 'M10', 'M11', 'M12', 'M13', 'M14', 'M15', 'M16', 'M17', 'M18', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'P1', 'P10', 'P11', 'P12', 'P13', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'S1', 'S10', 'S11', 'S12', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'V1', 'V10', 'V11', 'V12', 'V13', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'forward_returns', 'risk_free_rate', 'market_forward_excess_returns', 'lagged_forward_returns', 'lagged_risk_free_rate', 'lagged_market_forward_excess_returns', 'sp_index', 'ind_rsi_14', 'ind_rsi_42', 'ind_sma_5', 'ind_sma_30', 'ind_sma_200', 'ind_std_10', 'ind_std_50', 'ind_price_1y_ago', 'ind_bb_mid', 'ind_bb_std', 'ind_bb_uppe

In [6]:
import os
import warnings
warnings.filterwarnings("ignore")
import kaggle_evaluation.default_inference_server
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if __name__ == "__main__":
    if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        inference_server.serve()
    else:
        inference_server.run_local_gateway(("./kaggle/",))



[array([0.75562661]), array([0.86856882])]
[array([1.00186775]), array([1.42057931])]
[array([0.89539183]), array([0.47503274])]
[array([0.94234167]), array([1.91041773])]
[array([0.86056039]), array([1.74115961])]
[array([1.04579033]), array([0.9832411])]
[array([0.9183277]), array([1.19843714])]
[array([1.04427771]), array([1.77719583])]
[array([0.88127374]), array([0.93991366])]
[array([0.74823295]), array([0.])]


In [7]:
print(strategy.history_df.columns)
print(allpreds)

['date_id', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'E1', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'M1', 'M10', 'M11', 'M12', 'M13', 'M14', 'M15', 'M16', 'M17', 'M18', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'P1', 'P10', 'P11', 'P12', 'P13', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'S1', 'S10', 'S11', 'S12', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'V1', 'V10', 'V11', 'V12', 'V13', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'lagged_forward_returns', 'lagged_risk_free_rate', 'lagged_market_forward_excess_returns', 'sp_index']
[0.8120977161856252, 1.2112235287144124, 0.6852122851519556, 1.4263796988598354, 1.300859997495797, 1.0145157169641228, 1.0583824176346621, 1.4107367679700458, 0.9105937032926834, 0.3741164758675065]


In [10]:
import polars as pl
import numpy as np

# Global constant from your notebook
TRADING_DAYS_PER_YR = 252

def calculate_final_score(y_true_df: pl.DataFrame, signals: np.ndarray):
    """
    Calculates the final score for a given signal array against the true market returns.
    """
    # Convert Polars DataFrame to Pandas for easier indexing/calc if preferred, 
    # or stick to Polars. The provided snippet uses Pandas logic (iloc, etc).
    df_eval = y_true_df.to_pandas()
    
    # alignment check
    if len(signals) != len(df_eval):
        print(f"Warning: Signal length {len(signals)} != Eval length {len(df_eval)}. Truncating to shorter length.")
        min_len = min(len(signals), len(df_eval))
        df_eval = df_eval.iloc[:min_len]
        signals = signals[:min_len]
        
    df_eval['position'] = signals
    
    # Calculate Strategy Returns
    # Position 0 = 100% Risk Free
    # Position 1 = 100% Market (forward_returns)
    # Position >1 = Leveraged
    df_eval['strat_ret'] = (
        df_eval['risk_free_rate'] * (1 - df_eval['position']) +
        df_eval['position'] * df_eval['forward_returns']
    )
    
    # --- Metrics Calculation ---
    strat_excess = df_eval['strat_ret'] - df_eval['risk_free_rate']
    
    # Geometric Mean of Excess Returns
    strat_geo_mean = (1 + strat_excess).prod() ** (1 / len(df_eval)) - 1
    
    # Standard Deviation
    strat_std = df_eval['strat_ret'].std()
    
    if strat_std == 0:
        return 0.0, 0.0, 0.0, 0.0
    
    # Sharpe Ratio
    sharpe = (strat_geo_mean / strat_std) * np.sqrt(TRADING_DAYS_PER_YR)
    
    # Volatility Metrics
    mkt_std = df_eval['forward_returns'].std()
    mkt_vol = mkt_std * np.sqrt(TRADING_DAYS_PER_YR) * 100
    strat_vol = strat_std * np.sqrt(TRADING_DAYS_PER_YR) * 100
    
    # Volatility Penalty (Penalty if Vol > 1.2x Market Vol)
    excess_vol = max(0, strat_vol / mkt_vol - 1.2) if mkt_vol > 0 else 0
    vol_penalty = 1 + excess_vol
    
    # Return Gap Penalty (Penalty if Strategy < Market)
    mkt_excess = df_eval['forward_returns'] - df_eval['risk_free_rate']
    mkt_geo_mean = (1 + mkt_excess).prod() ** (1 / len(df_eval)) - 1
    
    return_gap_dual = (mkt_geo_mean - strat_geo_mean) * 100 * TRADING_DAYS_PER_YR
    return_gap = max(0, return_gap_dual)
    return_penalty = 1 + (return_gap**2) / 100
    
    final_score = sharpe / (vol_penalty * return_penalty)
    
    return final_score, return_gap_dual, mkt_vol, strat_vol

# ==========================================
# Execution Harness
# ==========================================

# 1. Load Data
# Assuming train_path is defined as in your notebook
df_full = pl.read_csv('./kaggle/train.csv')
if "date_id" in df_full.columns:
    df_full = df_full.sort("date_id")

# 2. Split Data strictly by time (Test = Last 180 Days)
# We replicate the logic from generate_test_csv to ensure alignment
days_to_keep = 180
unique_dates = df_full["date_id"].unique().sort()
cutoff_date = unique_dates.tail(days_to_keep).head(1).item()

# Train Set (History)
train_df = df_full.filter(pl.col("date_id") < cutoff_date)

# Ground Truth Test Set (Contains targets: forward_returns, risk_free_rate)
y_true_df = df_full.filter(pl.col("date_id") >= cutoff_date)


signals = allpreds

# 5. Calculate Score
# 'signals' corresponds to the prediction for the rows in 'y_true_df'
score, gap, mkt_vol, strat_vol = calculate_final_score(y_true_df, signals)

print(f"\nResults:")
print(f"Final Score: {score:.4f}")
print(f"Sharpe:      {score * (1 + max(0, strat_vol/mkt_vol - 1.2)) * (1 + max(0, gap)**2/100):.4f} (Approx raw sharpe)")
print(f"Market Vol:  {mkt_vol:.2f}%")
print(f"Strat Vol:   {strat_vol:.2f}%")
print(f"Return Gap:  {gap:.2f} bps")


Results:
Final Score: -0.1989
Sharpe:      -0.1989 (Approx raw sharpe)
Market Vol:  14.35%
Strat Vol:   16.96%
Return Gap:  -11.59 bps


In [9]:
import pandas as pd
dfs = pd.read_parquet('submission(1).parquet')
print(dfs['prediction'])

0    0.960248
1    1.299488
2    1.013143
3    0.925782
4    0.784164
5    0.860498
6    0.939799
7    0.850494
8    0.536229
9    0.758901
Name: prediction, dtype: float64
